In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import scipy.linalg as linalg
from src.pt import BrenierGaussian
from src.geom import log_map_gaussian
# change to latex font
plt.rcParams['mathtext.fontset'] = 'cm'

In [ ]:
# plot 2D Gaussian as heatmap

def plot_gaussians(mus, sigmas, xlim=(0, 10), ylim=(0, 10), resolution=100, labels=None):
    if not isinstance(mus, list):
        mus = [mus]
    if not isinstance(sigmas, list):
        sigmas = [sigmas]
    x = np.linspace(xlim[0], xlim[1], resolution)
    y = np.linspace(ylim[0], ylim[1], resolution)
    X, Y = np.meshgrid(x, y)
    zipped = zip(mus, sigmas)
    Z = np.sum([stats.multivariate_normal.pdf(np.dstack((X, Y)), mean=mu, cov=sigma) for mu, sigma in zipped], axis=0)
    plt.contourf(X, Y, Z, levels=10, cmap='viridis', vmin=0, vmax=0.35)
    # clip scale at 0.35, show contour plot
    if labels is not None:
        # add a dot on each mean with corresponding label
        # write the text in a box next to the dot
        for i, (mu, sigma) in enumerate(zip(mus, sigmas)):
            plt.scatter(mu[0], mu[1], label=labels[i], s=50, edgecolors='k')
            # make text thicker
            plt.text(mu[0]+0.2, mu[1]+0.2, labels[i], fontsize=15, bbox=dict(facecolor='white', alpha=0.7))
    # draw arrows from first mean to second mean
    plt.annotate('', xy=mus[1] - np.array([0,1.25]), xytext=mus[0] + np.array([0, 1.25]), arrowprops=dict(arrowstyle='->', color='white', lw=2))
    plt.annotate('', xy=mus[3] - np.array([0,0.75]), xytext=mus[2] + np.array([0, 1.25]), arrowprops=dict(arrowstyle='->', color='white', lw=2))
    plt.annotate('', xy=mus[2] + np.array([1.25,0]), xytext=mus[0] - np.array([1.25, 0]), arrowprops=dict(arrowstyle='->', color='grey', lw=2))
    # add text to middle of arrows
    plt.text((mus[0][0]+mus[1][0])/2 + 0.3, (mus[0][1]+mus[1][1])/2, r'$v$', fontsize=15, color='white', bbox=dict(facecolor='black', alpha=0.7))
    plt.text((mus[2][0]+mus[3][0])/2 + 0.3, (mus[2][1]+mus[3][1])/2, r'$\text{PT}_{\nu_i \to \mu_i^*}(v)$', fontsize=15, color='white', bbox=dict(facecolor='black', alpha=0.7))
    plt.text((mus[0][0]+mus[2][0])/2 - 0.25, (mus[0][1]+mus[2][1])/2 - 0.6, r'$\nabla \varphi$', fontsize=15, color='white', bbox=dict(facecolor='black', alpha=0.7))
    # force axes to be equal
    plt.axis('equal')
    # no axis ticks
    plt.xticks([])
    plt.yticks([])
    plt.xlim(xlim)
    plt.ylim(ylim)


### \nu_i ---> \nu_i+1

mean_nu1 = np.array([8, 2])
sigma_nu1 = np.array([[0.5, 0.2], [0.2, 0.5]])

mean_nu2 = np.array([8, 8])
sigma_nu2 = np.array([[0.5, -0.2], [-0.2, 0.5]])

mean_mu1 = np.array([2, 2])
sigma_mu1 = np.array([[0.5, -0.2], [-0.2, 0.5]])

A_nu_i, a_nu_i = log_map_gaussian(mean_nu1, sigma_nu1, mean_nu2, sigma_nu2)
brenier_map_nu_i = BrenierGaussian(mean_nu1, sigma_nu1, A_nu_i, a_nu_i)
# interpolate at 4 points between 0 and 1 and plot the resulting Gaussian distributions
means_tan, sigmas_tan = brenier_map_nu_i.interpolate(2)
fig, ax = plt.subplots(figsize=(6, 6))

# \nu_i ---> \mu_i^*
A_nu_i, a_nu_i = log_map_gaussian(mean_nu1, sigma_nu1, mean_mu1, sigma_mu1)
brenier_map_nu_mu = BrenierGaussian(mean_nu1, sigma_nu1, A_nu_i, a_nu_i)
means_geo, sigmas_geo = brenier_map_nu_mu.interpolate(2)

transported_As, transported_as = brenier_map_nu_i.parallel_transport(brenier_map_nu_mu, 100)
brenier_map_transported = BrenierGaussian(mean_mu1, sigma_mu1, transported_As[-1], transported_as[-1])
mean_transp_pushforward, sigma_transp_pushforward = brenier_map_transported.pushforward_measure(1)

means = [mean_nu1, mean_nu2, mean_mu1, mean_transp_pushforward]
sigmas = [sigma_nu1, sigma_nu2, sigma_mu1, sigma_transp_pushforward]
labels = [r'$\nu_i$', r'$\exp_{\nu_{i}}(\nabla\varphi)$', r'$\mu_i^*$', r'$\exp_{\mu_i^*}(\text{PT}_{\nu_i \to \mu_i^*}(\nabla \varphi))$']


plot_gaussians(means, sigmas, labels=labels, )
plt.savefig('gaussian_transport_1.png', dpi=1200)


In [ ]:
# plot 2D Gaussian as heatmap

def plot_gaussians(mus, sigmas, xlim=(0, 10), ylim=(0, 10), resolution=100, labels=None):
    if not isinstance(mus, list):
        mus = [mus]
    if not isinstance(sigmas, list):
        sigmas = [sigmas]
    x = np.linspace(xlim[0], xlim[1], resolution)
    y = np.linspace(ylim[0], ylim[1], resolution)
    X, Y = np.meshgrid(x, y)
    zipped = zip(mus, sigmas)
    Z = np.sum([stats.multivariate_normal.pdf(np.dstack((X, Y)), mean=mu, cov=sigma) for mu, sigma in zipped], axis=0)
    plt.contourf(X, Y, Z, levels=10, cmap='viridis', vmin=0, vmax=0.35)
    # clip scale at 0.35, show contour plot
    if labels is not None:
        # add a dot on each mean with corresponding label
        # write the text in a box next to the dot
        for i, (mu, sigma) in enumerate(zip(mus, sigmas)):
            plt.scatter(mu[0], mu[1], label=labels[i], s=50, edgecolors='k')
            # make text thicker
            plt.text(mu[0]+0.2, mu[1]+0.2, labels[i], fontsize=15, bbox=dict(facecolor='white', alpha=0.7))
    # draw arrows from first mean to second mean
    plt.annotate('', xy=mus[1] - np.array([0,1.25]), xytext=mus[0] + np.array([0, 1.25]), arrowprops=dict(arrowstyle='->', color='white', lw=2))
    plt.annotate('', xy=mus[3] - np.array([0,1.25]), xytext=mus[2] + np.array([0, 1.25]), arrowprops=dict(arrowstyle='->', color='white', lw=2))
    plt.annotate('', xy=mus[2] + np.array([1.1,-0.3]), xytext=mus[0] - np.array([1.15, -0.25]), arrowprops=dict(arrowstyle='->', color='grey', lw=2))
    # add text to middle of arrows
    plt.text((mus[0][0]+mus[1][0])/2 + 0.3, (mus[0][1]+mus[1][1])/2, r'$v$', fontsize=15, color='white', bbox=dict(facecolor='black', alpha=0.7))
    plt.text((mus[2][0]+mus[3][0])/2 + 0.3, (mus[2][1]+mus[3][1])/2-0.2, r'$\text{PT}_{\nu_i \to \mu_i^*}(v)$', fontsize=15, color='white', bbox=dict(facecolor='black', alpha=0.7))
    plt.text((mus[0][0]+mus[2][0])/2 - 0.45, (mus[0][1]+mus[2][1])/2 - 0.95, r'$\nabla \varphi$', fontsize=15, color='white', bbox=dict(facecolor='black', alpha=0.7))
    # force axes to be equal
    plt.axis('equal')
    # no axis ticks
    plt.xticks([])
    plt.yticks([])
    plt.xlim(xlim)
    plt.ylim(ylim)


### \nu_i ---> \nu_i+1

mean_nu1 = np.array([8, 2])
sigma_nu1 = np.array([[0.5, 0.2], [0.2, 0.5]])

mean_nu2 = np.array([8, 8])
sigma_nu2 = np.array([[0.5, 0.2], [0.2, 0.5]])

mean_mu1 = np.array([2, 4])
sigma_mu1 = np.array([[0.5, 0.2], [0.2, 0.5]])

A_nu_i, a_nu_i = log_map_gaussian(mean_nu1, sigma_nu1, mean_nu2, sigma_nu2)
brenier_map_nu_i = BrenierGaussian(mean_nu1, sigma_nu1, A_nu_i, a_nu_i)
# interpolate at 4 points between 0 and 1 and plot the resulting Gaussian distributions
means_tan, sigmas_tan = brenier_map_nu_i.interpolate(2)
fig, ax = plt.subplots(figsize=(6*11/12, 6))

# \nu_i ---> \mu_i^*
A_nu_i, a_nu_i = log_map_gaussian(mean_nu1, sigma_nu1, mean_mu1, sigma_mu1)
brenier_map_nu_mu = BrenierGaussian(mean_nu1, sigma_nu1, A_nu_i, a_nu_i)
means_geo, sigmas_geo = brenier_map_nu_mu.interpolate(2)

transported_As, transported_as = brenier_map_nu_i.parallel_transport(brenier_map_nu_mu, 100)
brenier_map_transported = BrenierGaussian(mean_mu1, sigma_mu1, transported_As[-1], transported_as[-1])
mean_transp_pushforward, sigma_transp_pushforward = brenier_map_transported.pushforward_measure(1)

means = [mean_nu1, mean_nu2, mean_mu1, mean_transp_pushforward]
sigmas = [sigma_nu1, sigma_nu2, sigma_mu1, sigma_transp_pushforward]
labels = [r'$\nu_i$', r'$\exp_{\nu_{i}}(\nabla\varphi)$', r'$\mu_i^*$', r'$\exp_{\mu_i^*}(\text{PT}_{\nu_i \to \mu_i^*}(\nabla \varphi))$']


plot_gaussians(means, sigmas, labels=labels, xlim=(0, 11), ylim=(0, 12))

plt.savefig('gaussian_transport_2.png', dpi=1200)
